# Macro indicator extraction for GAN conditioning

- load the Bloomberg export
- select a **small macro set**
- resample to **weekly**
- apply simple transformations
- export a clean macro feature matrix for the GAN

**NOT** using description-based search, Random Forest, or RFE.
We are selecting the macro indicators that are the best fit from the available Bloomberg series and that have usable history.


## Chosen indicators

These are the indicators selected for the first pass because they are broad, interpretable, and useful as external conditioning variables for generating synthetic data for the 30 assets.

### Labor / growth
- **INJCJC Index** — Initial Jobless Claims

### Policy / money market
- **FDFD Index** — Effective Fed Funds
- **US0003M Index** — USD 3M LIBOR

### Rates curve
- **USSWAP2 CMPN Curncy** — USD 2Y Swap
- **USSWAP10 CMPN Curncy** — USD 10Y Swap

### Broad commodity pressure
- **CRY Index** — CRB Commodity Index

### Credit / risk regime
- **IBOXUMAE CBIN Curncy** — Markit CDX IG
- **IBOXHYAE CBIN Curncy** — Markit CDX HY

In [15]:
import pandas as pd
import numpy as np

In [16]:
# File names
DATA_FILE = "../data/macro/bloomberg_data.csv"
DESC_FILE = "../data/macro/bloomberg_desc.csv" 

In [17]:
# ---------------------------------------------------
# 1. Load Bloomberg data
# ---------------------------------------------------
preamble = pd.read_csv(DATA_FILE, header=None, nrows=6, low_memory=False)

tickers = preamble.iloc[3, 1:].ffill().tolist()
fields = preamble.iloc[5, 1:].tolist()

cols = pd.MultiIndex.from_arrays([tickers, fields], names=["ticker", "field"])

raw = pd.read_csv(
    DATA_FILE,
    skiprows=6,
    header=None,
    index_col=0,
    parse_dates=True,
    low_memory=False
)

raw.index.name = "Date"
raw.columns = cols
raw = raw.sort_index()

# Keep only PX_LAST
px = raw.xs("PX_LAST", axis=1, level="field").copy()
px.columns.name = "ticker"

# Remove duplicate ticker columns
px = px.loc[:, ~px.columns.duplicated()]

print("PX_LAST shape:", px.shape)
print("Date range:", px.index.min(), "to", px.index.max())

PX_LAST shape: (9420, 292)
Date range: 1990-01-01 00:00:00 to 2026-02-06 00:00:00


In [18]:
# ---------------------------------------------------
# 2. Load descriptions
# ---------------------------------------------------
desc = pd.read_csv(DESC_FILE, header=None)
desc.columns = ["ticker", "name", "currency", "note"]
desc["ticker"] = desc["ticker"].astype(str).str.strip()

desc_map = desc.set_index("ticker")["name"].to_dict()

In [19]:

# ---------------------------------------------------
# 3. Hand-picked macro indicators
# ---------------------------------------------------
selected_tickers = [
    "INJCJC Index",            # Initial Jobless Claims
    "FDFD Index",              # Effective Fed Funds
    "US0003M Index",           # USD 3M LIBOR
    "USSWAP2 CMPN Curncy",     # USD 2Y Swap
    "USSWAP10 CMPN Curncy",    # USD 10Y Swap
    "CRY Index",               # CRB Commodity Index
    "IBOXUMAE CBIN Curncy",    # Markit CDX IG
    "IBOXHYAE CBIN Curncy",    # Markit CDX HY
]


available_tickers = [t for t in selected_tickers if t in px.columns]
missing_tickers = [t for t in selected_tickers if t not in px.columns]

print("Available selected tickers:", len(available_tickers))
print(available_tickers)

if missing_tickers:
    print("Missing selected tickers:")
    print(missing_tickers)


Available selected tickers: 8
['INJCJC Index', 'FDFD Index', 'US0003M Index', 'USSWAP2 CMPN Curncy', 'USSWAP10 CMPN Curncy', 'CRY Index', 'IBOXUMAE CBIN Curncy', 'IBOXHYAE CBIN Curncy']


In [20]:
macro_raw = px[available_tickers].copy()

macro_info = pd.DataFrame({
    "ticker": available_tickers,
    "description": [desc_map.get(t, "") for t in available_tickers],
    "missing_pct_daily": [macro_raw[t].isna().mean() for t in available_tickers]
}).sort_values("missing_pct_daily")

macro_info

,ticker,description,missing_pct_daily
2,US0003M Index,ICE LIBOR USD 3M,0.068153
3,USSWAP2 CMPN Curncy,US DOLLAR SWAP 2 YR,0.074098
4,USSWAP10 CMPN Curncy,US DOLLAR SWAP 10 YR,0.075053
5,CRY Index,FTSE/CoreCommodity CRB Excess,0.141189
1,FDFD Index,Fed Funds,0.175584
6,IBOXUMAE CBIN Curncy,MARKIT CDX IG,0.418577
7,IBOXHYAE CBIN Curncy,MARKIT CDX HY,0.529299
0,INJCJC Index,Initial Jobless Claims SA,0.800106


## Rename columns to simpler names

This makes the conditioning matrix easier to use later in the GAN pipeline.


In [ ]:

rename_map = {
    "INJCJC Index":           "initial_jobless_claims",
    "FDFD Index":             "fed_funds",
    "US0003M Index":          "usd_3m_libor",
    "USSWAP2 CMPN Curncy":    "usd_2y_swap",
    "USSWAP10 CMPN Curncy":   "usd_10y_swap",
    "CRY Index":              "crb_commodity_index",
    "IBOXUMAE CBIN Curncy":   "cdx_ig",
    "IBOXHYAE CBIN Curncy":   "cdx_hy",
}

macro_raw = macro_raw.rename(columns=rename_map)
macro_raw.head()


Columns after rename: ['initial_jobless_claims', 'fed_funds', 'usd_3m_libor', 'usd_2y_swap', 'usd_10y_swap', 'crb_commodity_index', 'cdx_ig', 'cdx_hy']


ticker,initial_jobless_claims,fed_funds,usd_3m_libor,usd_2y_swap,usd_10y_swap,crb_commodity_index,cdx_ig,cdx_hy
Date,,,,,,,,
1990-01-01,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-02,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-03,NaN,NaN,8.3750,NaN,NaN,NaN,NaN,NaN
1990-01-04,NaN,NaN,8.3125,8.44,8.89,NaN,NaN,NaN
1990-01-05,355.0,NaN,8.3750,8.43,8.92,NaN,NaN,NaN


## Weekly alignment

Macro indicators are easier to use as conditioning variables if everything is on one frequency.

In [22]:
macro_monthly = macro_raw.resample("W-FRI").last().ffill()

summary_monthly = pd.DataFrame({
    "feature": macro_monthly.columns,
    "missing_pct_monthly": macro_monthly.isna().mean().values,
    "first_valid": [macro_monthly[c].first_valid_index() for c in macro_monthly.columns],
    "last_valid": [macro_monthly[c].last_valid_index() for c in macro_monthly.columns]
}).sort_values("missing_pct_monthly")

summary_monthly

,feature,missing_pct_monthly,first_valid,last_valid
0,initial_jobless_claims,0.000000,1990-01-05,2026-02-06
2,usd_3m_libor,0.000000,1990-01-05,2026-02-06
3,usd_2y_swap,0.000000,1990-01-05,2026-02-06
4,usd_10y_swap,0.000000,1990-01-05,2026-02-06
5,crb_commodity_index,0.110934,1994-01-07,2026-02-06
1,fed_funds,0.141720,1995-02-17,2026-02-06
6,cdx_ig,0.408174,2004-10-01,2026-02-06
7,cdx_hy,0.493631,2007-11-02,2026-02-06



## Simple transformations

- Rates and credit spreads -> first difference
- Claims and commodity index -> percent change


In [23]:

def transform_feature(s):
    name = s.name

    # First difference: rates and credit spreads
    diff_features = [
        "fed_funds",
        "usd_3m_libor",
        "usd_2y_swap",
        "usd_10y_swap",
        "cdx_ig",
        "cdx_hy",
    ]

    # Percent change: claims and commodity index
    pct_features = [
        "initial_jobless_claims",
        "crb_commodity_index",
    ]

    if name in diff_features:
        return s.diff()

    if name in pct_features:
        return s.pct_change()

    return s.copy()


In [24]:
macro_features = macro_monthly.apply(transform_feature)
macro_features = macro_features.replace([np.inf, -np.inf], np.nan)

# Keep a recent usable sample window
macro_features = macro_features.loc["2005-01-31":].copy()

# Drop rows that are fully missing
macro_features = macro_features.dropna(how="all")

macro_features.head()

ticker,initial_jobless_claims,fed_funds,usd_3m_libor,usd_2y_swap,usd_10y_swap,crb_commodity_index,cdx_ig,cdx_hy
Date,,,,,,,,
2005-02-04,-0.072508,0.0000,0.02750,0.053,-0.059,-0.013752,-2.344,NaN
2005-02-11,0.003257,-0.1250,0.02438,0.034,0.001,0.018413,0.542,NaN
2005-02-18,0.032468,0.0625,0.05562,0.134,0.183,0.019730,-0.938,NaN
2005-02-25,-0.012579,0.0000,0.06000,0.067,0.007,0.037286,0.042,NaN
2005-03-04,0.060510,0.0000,0.04875,0.043,0.064,0.029175,-0.271,NaN


In [25]:
feature_quality = pd.DataFrame({
    "feature": macro_features.columns,
    "missing_pct": macro_features.isna().mean().values,
    "std": macro_features.std().values
}).sort_values(["missing_pct", "std"], ascending=[True, False])

feature_quality

,feature,missing_pct,std
6,cdx_ig,0.000000,7.052157
0,initial_jobless_claims,0.000000,0.297756
1,fed_funds,0.000000,0.282881
4,usd_10y_swap,0.000000,0.111774
3,usd_2y_swap,0.000000,0.097267
2,usd_3m_libor,0.000000,0.077192
5,crb_commodity_index,0.000000,0.024365
7,cdx_hy,0.131267,1.368214


## Keep the final feature set

Remove columns that are too incomplete.


In [26]:
MAX_MISSING = 0.20

final_features = feature_quality.loc[
    feature_quality["missing_pct"] <= MAX_MISSING, "feature"
].tolist()

macro_final = macro_features[final_features].dropna()

print("Final shape:", macro_final.shape)
print("Final features:")
print(macro_final.columns.tolist())

macro_final.head()

Final shape: (953, 8)
Final features:
['cdx_ig', 'initial_jobless_claims', 'fed_funds', 'usd_10y_swap', 'usd_2y_swap', 'usd_3m_libor', 'crb_commodity_index', 'cdx_hy']


ticker,cdx_ig,initial_jobless_claims,fed_funds,usd_10y_swap,usd_2y_swap,usd_3m_libor,crb_commodity_index,cdx_hy
Date,,,,,,,,
2007-11-09,8.434,0.018349,0.500,-0.0430,-0.1705,0.01438,0.002743,-1.360
2007-11-16,-1.157,-0.003003,0.000,-0.0575,-0.0285,0.06937,-0.014413,-0.245
2007-11-23,9.881,0.060241,0.125,-0.1395,-0.1795,0.09125,0.013908,-1.589
2007-11-30,-10.907,-0.022727,-0.125,-0.1740,-0.1260,0.09125,-0.040786,2.035
2007-12-07,0.231,-0.034884,-0.125,0.2355,0.1850,0.00938,0.009063,-0.290


In [27]:
# Correlation check
corr = macro_final.corr().round(2)
corr

ticker,cdx_ig,initial_jobless_claims,fed_funds,usd_10y_swap,usd_2y_swap,usd_3m_libor,crb_commodity_index,cdx_hy
ticker,,,,,,,,
cdx_ig,1.00,0.17,0.03,-0.23,-0.11,0.07,-0.36,-0.62
initial_jobless_claims,0.17,1.00,-0.14,-0.06,-0.01,0.16,-0.16,-0.23
fed_funds,0.03,-0.14,1.00,-0.03,-0.08,0.06,0.03,0.00
usd_10y_swap,-0.23,-0.06,-0.03,1.00,0.70,0.16,0.25,0.23
usd_2y_swap,-0.11,-0.01,-0.08,0.70,1.00,0.33,0.11,0.08
usd_3m_libor,0.07,0.16,0.06,0.16,0.33,1.00,-0.03,-0.02
crb_commodity_index,-0.36,-0.16,0.03,0.25,0.11,-0.03,1.00,0.35
cdx_hy,-0.62,-0.23,0.00,0.23,0.08,-0.02,0.35,1.00


In [28]:
# Export final features
macro_final.to_csv("../data/macro/macro_conditioning_features.csv")

print("Saved: macro_conditioning_features.csv")

Saved: macro_conditioning_features.csv
